## Conditions on aggregats (HAVING)

The following cell creates table that would be used as example.

In [16]:
--postgreSQL
DROP TABLE IF EXISTS aggregation_table;
CREATE TABLE aggregation_table(
    col1 TEXT,
    col2 INT
);
INSERT INTO aggregation_table(col1, col2) VALUES
('A', 5),
('A', 1),
('B', 2),
('B', 1),
('C', 3),
('C', 4);

DROP TABLE
CREATE TABLE
INSERT 0 6


Let's begin with the view of the example table.

In [12]:
--postgreSQL
SELECT * FROM aggregation_table;

SELECT 6


col1,col2
A,5
A,1
B,2
B,1
C,3
C,4


Now a problem: I need to aggregate `SUM(col2)` by the values of `col1` and I only get the results where the sums are greater than 5. So **I need to set a condition on a result of the aggregation.**

## Won't work

The first thing that comes to mind is to use the arger functions inside the `WHERE` block. This will cause an error because the `WHERE` block in sql is executed before all aggregations.

In [ ]:
--postgreSQL
SELECT col1, SUM(col2)
FROM aggregation_table 
WHERE SUM(col2) > 5
GROUP BY col1;

Traceback (most recent call last):
  File "/home/fedor/Documents/code/knowledge/src/sql_kernel/kernel.py", line 116, in do_execute
    self._execute_sql(code=code)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^
  File "/home/fedor/Documents/code/knowledge/src/sql_kernel/kernel.py", line 84, in _execute_sql
    messages, tables = runner.execute(code)
                       ~~~~~~~~~~~~~~^^^^^^
  File "/home/fedor/Documents/code/knowledge/src/sql_kernel/runners/runners.py", line 73, in execute
    raise e
  File "/home/fedor/Documents/code/knowledge/src/sql_kernel/runners/runners.py", line 70, in execute
    cursor.execute(code.encode())
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "/home/fedor/.virtualenvironments/knowledge/lib/python3.13/site-packages/psycopg/cursor.py", line 97, in execute
    raise ex.with_traceback(None)
psycopg.errors.GroupingError: aggregate functions are not allowed in WHERE
LINE 4: WHERE SUM(col2) > 5
              ^


## Subquery

A possible but **not optimal solution** is to use aggregation in the subquery and then describe the condition on the aggregation in the external query.

---

In the following example the problem is solved using subquery

In [17]:
--postgreSQL
SELECT * FROM (
    SELECT col1, SUM(col2)
    FROM aggregation_table
    GROUP BY col1
) AS tab1
WHERE sum > 5;

SELECT 2


col1,sum
C,7
A,6


## `HAVING`

There's a special keyword for describing these cases: `HAVING`, which is the same as `WHERE`, but for aggregates. **It's the optimal way to solve such a task.**

You have to use `HAVING` after `GROUP BY` satement.

---

The following example applies the condition on the result of the aggregation function.

In [18]:
--postgreSQL
SELECT col1, SUM(col2)
FROM aggregation_table
GROUP BY col1
HAVING SUM(col2) > 5;

SELECT 2


col1,sum
C,7
A,6


**Note** You can also use conditions on aggregates not mentioned in the `SELECT` statement. So in the next cell I got sums of `col2` by `col1`, but only for cases where the average of `col2` by `col1` is greater than 3.

In [19]:
--postgreSQL
SELECT col1, SUM(col2)
FROM aggregation_table
GROUP BY col1
HAVING AVG(col2) > 3;

SELECT 1


col1,sum
C,7


**Note** You can use aggregation variables in conditions. So in the following example I got sums of `col2` only for certain values of `col1`.

In [20]:
--postgreSQL
SELECT col1, SUM(col2)
FROM aggregation_table
GROUP BY col1
HAVING col1 IN ('A', 'B');

SELECT 2


col1,sum
B,3
A,6
